# Исправление get_atm()

## Проверка как можно избежать транспонирования матрицы в функции get_atm

In [1]:
import tracemalloc

import numpy as np
import pandas as pd

In [2]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
d = {}

s1 = pd.Series({1:'a', 2:'b'})
s2 = pd.Series({3:'c', 4:'d'})
s3 = pd.Series({1:'a1', 3:'c1'})
d['q'] = s1
d['w'] = s2
d['e'] = s3

df = pd.DataFrame(d).T

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df

Текущее использование: 0.03 MB
Пиковое использование: 0.04 MB
CPU times: total: 15.6 ms
Wall time: 28 ms


,1,2,3,4
q,a,b,NaN,NaN
w,NaN,NaN,c,d
e,a1,NaN,c1,NaN


#### Вариант 1

In [3]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
df2 = pd.DataFrame()

dft = pd.DataFrame()
dft['q'] = s1
df2 = pd.concat([df2, dft.T])

dft = pd.DataFrame()
dft['w'] = s2
df2 = pd.concat([df2, dft.T])

dft = pd.DataFrame()
dft['e'] = s3
df2 = pd.concat([df2, dft.T])

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df2

Текущее использование: 0.04 MB
Пиковое использование: 0.06 MB
CPU times: total: 31.2 ms
Wall time: 36 ms


,1,2,3,4
q,a,b,NaN,NaN
w,NaN,NaN,c,d
e,a1,NaN,c1,NaN


#### Вариант 2

In [4]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
d = {}

d['q'] = s1
d['w'] = s2
d['e'] = s3

# Вместо транспонирования:
df3 = pd.DataFrame.from_dict(d, orient='index')

# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df3


Текущее использование: 0.01 MB
Пиковое использование: 0.01 MB
CPU times: total: 0 ns
Wall time: 4 ms


,1,2,3,4
q,a,b,NaN,NaN
e,a1,NaN,c1,NaN
w,NaN,NaN,c,d


Пожалуй это идеальное решение!

#### Вариант 3

In [5]:
%%time

# Начать отслеживание
tracemalloc.start()

# Код:
df4 = pd.DataFrame()

dft = pd.DataFrame.from_dict({'q': {1:'a', 2:'b'}}, orient='index')
df4 = pd.concat([df4, dft])
dft = pd.DataFrame.from_dict({'w': {3:'c', 4:'d'}}, orient='index')
df4 = pd.concat([df4, dft])
dft = pd.DataFrame.from_dict({'e': {1:'a1', 3:'c1', 5:'e'}}, orient='index')
df4 = pd.concat([df4, dft])


# Получить статистику
current, peak = tracemalloc.get_traced_memory()
print(f"Текущее использование: {current / 1024 / 1024:.2f} MB")
print(f"Пиковое использование: {peak / 1024 / 1024:.2f} MB")

# Остановить отслеживание
tracemalloc.stop()

# Печать результата - датафрейма
df4

Текущее использование: 0.02 MB
Пиковое использование: 0.04 MB
CPU times: total: 46.9 ms
Wall time: 36 ms


,1,2,3,4,5
q,a,b,NaN,NaN,NaN
w,NaN,NaN,c,d,NaN
e,a1,NaN,c1,NaN,e


Достоинство данного подхода в том, что не забивается память. На каждой итерации данные добавляются сразу в матрицу, а не накапливаются в словаре или списке.